# Train LSTM Brick Predictor (Colab GPU)
1. Загрузи `aip/` в Google Drive
2. Запусти ячейки по порядку
3. Скачай model.tflite + vocab.json + model_metadata.json
4. Скопируй их в `catroid/src/main/assets/`

In [ ]:
import os, sys, json, time, shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# @title Настройки
# После unzip colab_pack.zip файлы лежат в /content/
POSSIBLE_PATHS = [
    '/content/training_data/projects.json',
    '/content/aip/training_data/projects.json'
]
PROJECTS_JSON = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        PROJECTS_JSON = p
        break
MODEL_DIR = '/content/model'

WINDOW = 500
EPOCHS = 15
BATCH_SIZE = 128
EMBED_DIM = 128
LSTM_UNITS = 256

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# @title Загрузить projects.json
from google.colab import files
if PROJECTS_JSON is None or not os.path.exists(PROJECTS_JSON):
    print('Загрузи projects.json (из colab_pack.zip):')
    uploaded = files.upload()
    for fn in uploaded.keys():
        PROJECTS_JSON = '/content/' + fn
        print(f'Сохранён: {PROJECTS_JSON}')

with open(PROJECTS_JSON, 'r', encoding='utf-8') as f:
    projects = json.load(f)
num_projects = len(projects)
print(f'Загружено проектов: {num_projects}')

In [ ]:
# @title Собрать словарь
from collections import Counter

SPECIAL = ['[PAD]', '[UNK]', '[SEP]', '[START]', '[END]']
PAD_ID, UNK_ID, SEP_ID, START_ID, END_ID = 0, 1, 2, 3, 4

counter = Counter()
for proj in projects:
    for scene in proj.get('scenes', []):
        for sprite in scene.get('sprites', []):
            for script in sprite.get('scripts', []):
                for brick in script.get('bricks', []):
                    bt = brick.get('type', '')
                    if bt: counter[bt] += 1

word2id = {tok: i for i, tok in enumerate(SPECIAL)}
for word, freq in counter.most_common():
    if word not in word2id: word2id[word] = len(word2id)
id2word = {v: k for k, v in word2id.items()}
vocab_size = len(word2id)
print(f'Vocabulary size: {vocab_size}')

In [ ]:
# @title Генерация training pairs
def encode(bt): return word2id.get(bt, UNK_ID)

def build_project_sequence(proj, max_len=1500):
    tokens = [START_ID]
    for scene in proj.get('scenes', []):
        for sprite in scene.get('sprites', []):
            for script in sprite.get('scripts', []):
                for brick in script.get('bricks', []):
                    bt = brick.get('type', '')
                    if bt: tokens.append(encode(bt))
                tokens.append(SEP_ID)
            tokens.append(SEP_ID)
        tokens.append(SEP_ID)
    tokens.append(END_ID)
    return tokens[-max_len:]

inputs, targets = [], []
for proj in projects:
    all_tokens = build_project_sequence(proj, WINDOW * 3)
    for i in range(WINDOW, len(all_tokens)):
        ctx = all_tokens[i - WINDOW:i]
        tgt = all_tokens[i]
        if tgt in (PAD_ID, START_ID): continue
        inputs.append(ctx)
        targets.append(tgt)

import gc; del projects; gc.collect()

X = np.array(inputs, dtype=np.int32)
y = np.array(targets, dtype=np.int32)
del inputs, targets; gc.collect()

print(f'Training pairs: {len(X)}')
print(f'Input shape: {X.shape}')

In [ ]:
# @title Build & Train LSTM
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, EMBED_DIM, mask_zero=True),
    tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=False, dropout=0.3),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

t0 = time.time()
history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_split=0.1, verbose=1)
train_time = time.time() - t0
print(f'\nTraining time: {train_time:.1f}s')
print(f'Final acc: {history.history["accuracy"][-1]:.4f}')

In [ ]:
# @title Convert to TFLite + Save
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(MODEL_DIR, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
size_mb = len(tflite_model) / (1024*1024)
print(f'Model: {tflite_path} ({size_mb:.2f} MB)')

# vocab.json
vocab_data = {'word2id': word2id,
              'id2word': {str(k): v for k,v in id2word.items()},
              'vocab_size': vocab_size}
with open(os.path.join(MODEL_DIR, 'vocab.json'), 'w') as f:
    json.dump(vocab_data, f, indent=2)

# metadata
metadata = {
    'vocab_size': vocab_size,
    'window': WINDOW,
    'model_version': 3,
    'brick_types': list(word2id.keys()),
    'training_projects': num_projects,
    'training_pairs': len(X),
    'epochs': EPOCHS,
    'final_accuracy': round(float(history.history['accuracy'][-1]), 4)
}
with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('\n=== ГОТОВО ===')
print(f'{tflite_path}')
print(f'{os.path.join(MODEL_DIR, "vocab.json")}')
print(f'{os.path.join(MODEL_DIR, "model_metadata.json")}')

In [ ]:
# @title Скачать результаты
from google.colab import files
files.download(tflite_path)
files.download(os.path.join(MODEL_DIR, 'vocab.json'))
files.download(os.path.join(MODEL_DIR, 'model_metadata.json'))